In [1]:
import pandas as pd

# Define relative path to the raw dataset
file_path = "../data/raw/DDoS-Friday-no-metadata.parquet"

# Load the parquet file into a pandas DataFrame
df = pd.read_parquet(file_path)

# Strip leading and trailing whitespace from column headers
df.columns = df.columns.str.strip()

# Display dataset dimensions (rows, columns)
print("--- Dataset Dimensions ---")
print(f"Total Rows (Flows): {df.shape[0]:,}")
print(f"Total Features (Columns): {df.shape[1]}")

# Display distribution of target labels (Normal vs Attack)
print("\n--- Target Classes (Label Breakdown) ---")
print(df['Label'].value_counts())

# Preview first 5 rows
df.head()

--- Dataset Dimensions ---
Total Rows (Flows): 221,264
Total Features (Columns): 78

--- Target Classes (Label Breakdown) ---
Label
DDoS      128014
Benign     93250
Name: count, dtype: int64


,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,6,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
1,6,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
2,6,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
3,6,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
4,6,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign


In [2]:
# Select the top 10 most critical network flow features
key_features = [
    'Flow Duration', 
    'Total Fwd Packets', 
    'Total Backward Packets',
    'Total Length of Fwd Packets', 
    'Fwd Packet Length Max',
    'Flow Bytes/s', 
    'Flow Packets/s', 
    'SYN Flag Count', 
    'ACK Flag Count', 
    'Label'
]

# Filter existing columns to avoid key errors if names differ slightly
available_keys = [col for col in key_features if col in df.columns]

# Print statistical summary of the key features
print("--- Key Features Statistical Summary ---")
df[available_keys].describe().T

--- Key Features Statistical Summary ---


,count,mean,std,min,25%,50%,75%,max
Flow Duration,221264.0,1.657047e+07,3.175633e+07,-1.0,90322.000000,1.614922e+06,9.018368e+06,1.199999e+08
Total Fwd Packets,221264.0,4.931069e+00,1.557175e+01,1.0,2.000000,3.000000e+00,5.000000e+00,1.932000e+03
Total Backward Packets,221264.0,4.647882e+00,2.196767e+01,0.0,1.000000,4.000000e+00,6.000000e+00,2.942000e+03
Fwd Packet Length Max,221264.0,5.486506e+02,1.881396e+03,0.0,6.000000,2.000000e+01,3.500000e+01,1.168000e+04
Flow Bytes/s,221264.0,2.732280e+05,8.222828e+06,-12000000.0,11.718325,8.158733e+02,1.870054e+04,2.070000e+09
Flow Packets/s,221264.0,8.107003e+03,8.280048e+04,-2000000.0,0.606072,4.883782e+00,6.092128e+01,3.000000e+06
SYN Flag Count,221264.0,3.116639e-02,1.737676e-01,0.0,0.000000,0.000000e+00,0.000000e+00,1.000000e+00
ACK Flag Count,221264.0,5.038416e-01,4.999864e-01,0.0,0.000000,1.000000e+00,1.000000e+00,1.000000e+00


In [3]:
import numpy as np

# Create a copy to preserve raw data
df_clean = df.copy()

# Replace infinite values with NaN
df_clean.replace([np.inf, -np.inf], np.nan, inplace=True)

# Drop rows where critical flow metrics contain negative numbers or NaNs
invalid_condition = (
    (df_clean['Flow Duration'] < 0) | 
    (df_clean['Flow Bytes/s'] < 0) | 
    (df_clean['Flow Packets/s'] < 0)
)
df_clean = df_clean[~invalid_condition]
df_clean.dropna(inplace=True)

# Drop zero-variance (constant) columns where all rows have the exact same value
constant_cols = [col for col in df_clean.columns if df_clean[col].nunique() <= 1]
df_clean.drop(columns=constant_cols, inplace=True)

print(f"Original Rows: {df.shape[0]:,}")
print(f"Cleaned Rows: {df_clean.shape[0]:,}")
print(f"Rows Dropped: {df.shape[0] - df_clean.shape[0]:,}")
print(f"Constant Columns Removed ({len(constant_cols)}): {constant_cols}")

C:\Users\Aniket\AppData\Local\Temp\ipykernel_12900\1535329743.py:16: Pandas4Warning: The inplace keyword in DataFrame.dropna is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_clean.dropna(inplace=True)


Original Rows: 221,264
Cleaned Rows: 221,262
Rows Dropped: 2
Constant Columns Removed (10): ['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'CWE Flag Count', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']


C:\Users\Aniket\AppData\Local\Temp\ipykernel_12900\1535329743.py:20: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_clean.drop(columns=constant_cols, inplace=True)


In [4]:
import os

# Ensure the processed directory exists
output_dir = "../data/processed"
os.makedirs(output_dir, exist_ok=True)

output_file = os.path.join(output_dir, "cleaned_ddos_flows.parquet")

# Export cleaned dataframe to a compressed parquet file
df_clean.to_parquet(output_file, index=False)
print(f"Cleaned dataset successfully saved to: {output_file}")

Cleaned dataset successfully saved to: ../data/processed\cleaned_ddos_flows.parquet
